# LeetCode 727 — Minimum Window Subsequence bằng Python

- Link: https://leetcode.com/problems/minimum-window-subsequence/
- Độ khó: **Hard**
- Chủ đề: Chuỗi, Hai con trỏ (two pointers), Quy hoạch động (DP), Bảng "vị trí xuất hiện kế tiếp" (next-occurrence table), Sliding window

> **Ghi chú Premium:** đây là bài **Premium** trên LeetCode. Đề bài dưới đây được viết lại theo hiểu biết về bài; nếu bạn có tài khoản Premium, hãy đối chiếu lại câu chữ, ví dụ và ràng buộc trên trang gốc.

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minWindow(self, s1: str, s2: str) -> str:
```

## 1. Giới thiệu & mục tiêu học

Bài này là "anh em" của *Minimum Window Substring (76)*, nhưng khác một điểm then chốt: ở bài 76 ta chỉ cần cửa sổ chứa đủ **các ký tự** (không quan tâm thứ tự), còn ở đây cửa sổ phải chứa `s2` như một **dãy con (subsequence)** — tức là **đúng thứ tự**. Chính ràng buộc thứ tự làm cho kỹ thuật sliding window với bộ đếm của bài 76 **không còn dùng được**.

Sau notebook này bạn sẽ:

1. Phân biệt rõ **substring** (đoạn con liên tiếp) và **subsequence** (dãy con giữ thứ tự, có thể bỏ ký tự).
2. Viết brute force kiểm tra mọi cửa sổ, rồi cải tiến bằng **tham lam từ mỗi điểm bắt đầu**.
3. Hiểu kỹ thuật **quét xuôi – quét ngược (forward–backward)** để thu hẹp cửa sổ.
4. Viết lời giải **quy hoạch động** `dp[j]` = điểm bắt đầu muộn nhất, và lời giải dùng **bảng vị trí kế tiếp**.
5. Xử lý đúng yêu cầu **"nhiều cửa sổ cùng độ dài thì lấy cửa sổ bên trái nhất"**.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho hai chuỗi `s1` và `s2`. Tìm **đoạn con liên tiếp** `W` ngắn nhất của `s1` sao cho `s2` là **dãy con** của `W`.

- Nếu không có cửa sổ nào như vậy, trả về chuỗi rỗng `""`.
- Nếu có nhiều cửa sổ cùng độ dài nhỏ nhất, trả về cửa sổ có **chỉ số bắt đầu nhỏ nhất** (bên trái nhất).

**Ví dụ 1**

```text
Input:  s1 = "abcdebdde", s2 = "bde"
Output: "bcde"
Giải thích: "bcde" chứa b..d..e theo thứ tự. "bdde" cũng hợp lệ và cùng độ dài 4,
nhưng "bcde" xuất hiện trước. "deb" không hợp lệ vì thứ tự b, d, e bị sai.
```

**Ví dụ 2**

```text
Input:  s1 = "jmeqksfrsdcmsiwvaovztaqenprpvnbstl", s2 = "u"
Output: ""
```

**Ràng buộc** (theo hiểu biết):

- `1 <= s1.length <= 2 * 10^4`
- `1 <= s2.length <= 100`
- `s1`, `s2` chỉ gồm chữ cái tiếng Anh thường.

In [1]:
EXAMPLES = [
    ("abcdebdde", "bde", "bcde"),
    ("jmeqksfrsdcmsiwvaovztaqenprpvnbstl", "u", ""),
]
for s1, s2, exp in EXAMPLES:
    print(f"s1={s1!r}, s2={s2!r} -> mong đợi {exp!r}")

s1='abcdebdde', s2='bde' -> mong đợi 'bcde'
s1='jmeqksfrsdcmsiwvaovztaqenprpvnbstl', s2='u' -> mong đợi ''


## 3. Kiến thức nền

### 3.1. Substring và subsequence

- **Substring** `s[i:j]`: các ký tự **liên tiếp**.
- **Subsequence**: chọn một số ký tự, **giữ nguyên thứ tự**, được phép bỏ qua ký tự ở giữa.

Ví dụ với `"abcde"`: `"bcd"` là cả substring lẫn subsequence; `"ace"` chỉ là subsequence; `"ca"` không là gì cả.

In [2]:
def is_subsequence(small, big):
    """Kiểm tra small có phải dãy con của big không (tham lam, O(len(big)))."""
    j = 0
    for ch in big:
        if j < len(small) and ch == small[j]:
            j += 1
    return j == len(small)

for t in ["bcd", "ace", "ca", ""]:
    print(f"{t!r:6} subsequence của 'abcde'? {is_subsequence(t, 'abcde')}")

'bcd'  subsequence của 'abcde'? True
'ace'  subsequence của 'abcde'? True
'ca'   subsequence của 'abcde'? False
''     subsequence của 'abcde'? True


### 3.2. Vì sao kiểm tra dãy con bằng tham lam là đúng?

Khi gặp ký tự trùng `small[j]`, ta **dùng ngay** nó. Dùng sớm không bao giờ thiệt: nếu có cách khớp dùng một vị trí muộn hơn cho `small[j]`, ta có thể đổi sang vị trí sớm hơn mà các ký tự sau vẫn khớp được. Đây là lập luận "trao đổi" (exchange argument) rất hay gặp.

### 3.3. Mẹo `iter` + `in` để kiểm tra dãy con

`ch in it` với `it` là iterator sẽ **tiêu thụ** iterator đến khi gặp `ch`. Vì vậy `all(ch in it for ch in small)` là một cách kiểm tra dãy con rất gọn.

In [3]:
def is_subseq_iter(small, big):
    it = iter(big)
    return all(ch in it for ch in small)

print(is_subseq_iter("ace", "abcde"), is_subseq_iter("ca", "abcde"))

True False


### 3.4. So sánh "ngắn hơn, nếu bằng thì bên trái hơn"

Lưu đáp án dạng `(độ_dài, start)`. Tuple so sánh theo thứ tự từ điển: độ dài trước, start sau → `min` tự động chọn đúng.

In [4]:
candidates = [(4, 5), (4, 1), (6, 0)]
print(min(candidates))   # (4, 1): độ dài 4, bắt đầu sớm nhất

(4, 1)


### 3.5. Bảng `float('inf')` và `-1` làm giá trị "không tồn tại"

Trong DP, ta hay dùng `-1` để nói "chưa có điểm bắt đầu hợp lệ", và `float('inf')` cho "độ dài vô cùng". Nhớ kiểm tra trước khi dùng làm chỉ số!

In [5]:
best_len = float('inf')
print(best_len > 10**9, 5 < best_len)
start = -1
print("có cửa sổ?" , start != -1)

True True
có cửa sổ? False


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Một cửa sổ "tối thiểu cục bộ" trông thế nào?

Nếu `W = s1[l..r]` là cửa sổ ngắn nhất, thì:

- `s1[l] == s2[0]` (nếu không, bỏ `s1[l]` vẫn còn chứa `s2` → ngắn hơn).
- `s1[r] == s2[-1]` (tương tự).

Vậy ta chỉ cần thử các điểm bắt đầu có ký tự `s2[0]`.

### 4.2. Tính tay `s1 = "abcdebdde"`, `s2 = "bde"`

Chỉ số: `a0 b1 c2 d3 e4 b5 d6 d7 e8`.

| start (ký tự `b`) | Khớp tham lam b → d → e | end | Cửa sổ | Độ dài |
|---|---|---|---|---|
| 1 | b@1, d@3, e@4 | 4 | `bcde` | 4 |
| 5 | b@5, d@6, e@8 | 8 | `bdde` | 4 |

Hai cửa sổ cùng dài 4 → chọn start nhỏ hơn: `bcde`. ✔

### 4.3. Vì sao không dùng sliding window "co giãn" như bài 76?

Ở bài 76, khi cửa sổ hợp lệ, ta co trái từng bước và cập nhật bộ đếm. Ở đây "hợp lệ" phụ thuộc **thứ tự**, không có bộ đếm đơn giản nào cho biết bỏ ký tự trái có còn hợp lệ không. Ta cần cách khác: **từ điểm kết thúc, quét ngược để tìm điểm bắt đầu muộn nhất**.

In [6]:
s1, s2 = "abcdebdde", "bde"
for start in [i for i, c in enumerate(s1) if c == s2[0]]:
    j, end = 0, None
    for k in range(start, len(s1)):
        if s1[k] == s2[j]:
            j += 1
            if j == len(s2):
                end = k
                break
    print("start", start, "end", end, "->", s1[start:end + 1] if end is not None else None)

start 1 end 4 -> bcde
start 5 end 8 -> bdde


## 5. Cách 1 — Brute Force: thử mọi cửa sổ

**Ý tưởng.** Duyệt mọi cặp `(l, r)`, kiểm tra `s2` có là dãy con của `s1[l:r+1]` không, giữ cửa sổ ngắn nhất (ưu tiên trái). Duyệt `l` tăng dần và chỉ cập nhật khi **ngắn hơn hẳn** → tự động ưu tiên bên trái.

In [7]:
def min_window_brute(s1, s2):
    n = len(s1)
    best = (float('inf'), -1)
    for l in range(n):
        for r in range(l, n):
            if r - l + 1 >= best[0]:
                break                       # không thể tốt hơn
            if is_subsequence(s2, s1[l:r + 1]):
                best = (r - l + 1, l)
                break                       # r nhỏ nhất cho l này
    return "" if best[1] == -1 else s1[best[1]:best[1] + best[0]]

for s1, s2, exp in EXAMPLES:
    print(min_window_brute(s1, s2), min_window_brute(s1, s2) == exp)

bcde True
 True


**Độ phức tạp.** `O(n^2)` cửa sổ × `O(n)` kiểm tra = `O(n^3)` trong trường hợp xấu. Với `n = 2·10^4` là `8·10^12` phép — không thể.

**Nút thắt.** Với một `l` cố định, việc tìm `r` nhỏ nhất chỉ cần **một lần quét tham lam** từ `l` (mục 4.2) — không cần thử từng `r` rồi kiểm tra lại từ đầu.

## 6. Bước đột phá

**Insight 1 — tham lam từ mỗi điểm bắt đầu.** Với `l` cố định, `r` nhỏ nhất là vị trí kết thúc của phép khớp tham lam `s2` bắt đầu từ `l`. → `O(n)` cho mỗi `l`, tổng `O(n^2)` (Cách 2).

**Insight 2 — quét ngược để thu hẹp.** Khi khớp xuôi tới `end`, **quét ngược** từ `end` khớp `s2` từ cuối về đầu, ta được `start` **muộn nhất** sao cho `s1[start..end]` vẫn chứa `s2`. Đó là cửa sổ tốt nhất kết thúc tại `end`. Sau đó tiếp tục quét xuôi từ `start + 1` (Cách 3).

**Insight 3 — DP theo điểm kết thúc.** Định nghĩa `dp[j]` = chỉ số bắt đầu **muộn nhất** `st` sao cho `s2[0..j]` là dãy con của `s1[st..i]` (với `i` là vị trí đang xét). Khi gặp `s1[i] == s2[j]`: `dp[j] = dp[j-1]` (với `j = 0`: `dp[0] = i`). Mỗi khi `dp[m-1]` được cập nhật, ta có cửa sổ `[dp[m-1], i]` (Cách 4).

**Insight 4 — nhảy bằng bảng vị trí kế tiếp.** `nxt[i][c]` = vị trí nhỏ nhất `≥ i` có ký tự `c`. Khớp tham lam từ `l` chỉ cần `m` lần tra bảng (không quét từng ký tự) → `O(n·m)` với hằng số nhỏ (Cách 5).

**Quy tắc nếu… thì…**

- Nếu `s1[i] == s2[j]` thì chuỗi khớp tiến một bước.
- Nếu khớp đủ `m` ký tự tại `end` thì quét ngược để tìm `start` muộn nhất, so sánh `(end - start + 1, start)`.
- Nếu hai cửa sổ cùng độ dài thì giữ cửa sổ có start nhỏ hơn → chỉ cập nhật khi **nhỏ hơn hẳn** và duyệt theo thứ tự tăng.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Tham lam từ mỗi điểm bắt đầu `O(n·n)`

**Giải thích.** Với mỗi `l` mà `s1[l] == s2[0]`, quét xuôi khớp tham lam; nếu khớp đủ tại `r` thì cập nhật đáp án. Có thể dừng sớm khi `r - l + 1 >= best_len`.

In [8]:
def min_window_greedy_start(s1, s2):
    n, m = len(s1), len(s2)
    best_len, best_l = float('inf'), -1
    for l in range(n):
        if s1[l] != s2[0]:
            continue
        j = 0
        for r in range(l, n):
            if r - l + 1 > best_len:      # dài hơn kỷ lục -> bỏ
                break
            if s1[r] == s2[j]:
                j += 1
                if j == m:
                    if r - l + 1 < best_len:
                        best_len, best_l = r - l + 1, l
                    break
    return "" if best_l == -1 else s1[best_l:best_l + best_len]

for s1, s2, exp in EXAMPLES:
    print(min_window_greedy_start(s1, s2) == exp)

True
True


**Code nộp LeetCode (Cách 2)** — đúng nhưng có thể TLE với `s1 = "aaaa...a"`, `s2 = "aab"` (mỗi `l` quét tới cuối).

```python
class Solution:
    def minWindow(self, s1: str, s2: str) -> str:
        n, m = len(s1), len(s2)
        best_len, best_l = float('inf'), -1
        for l in range(n):
            if s1[l] != s2[0]:
                continue
            j = 0
            for r in range(l, n):
                if r - l + 1 > best_len:
                    break
                if s1[r] == s2[j]:
                    j += 1
                    if j == m:
                        if r - l + 1 < best_len:
                            best_len, best_l = r - l + 1, l
                        break
        return '' if best_l == -1 else s1[best_l:best_l + best_len]
```

**Độ phức tạp.** Time `O(n^2)` trường hợp xấu, Space `O(1)`.

### 7.2. Cách 3 — Quét xuôi rồi quét ngược (two pointers)

**Giải thích.**

1. Con trỏ `i` chạy trên `s1`, `j` trên `s2`. Mỗi khi `s1[i] == s2[j]` thì `j += 1`.
2. Khi `j == m` → `end = i`. Đặt `j = m - 1`, quét **ngược** `i` từ `end`: mỗi khi `s1[i] == s2[j]` thì `j -= 1`; khi `j < 0` → `start = i`.
3. Cập nhật đáp án với `[start, end]`.
4. Đặt `i = start + 1`, `j = 0` và lặp lại.

Tại sao quay về `start + 1` mà không phải `end + 1`? Vì có thể có cửa sổ **bắt đầu trong `(start, end]`** và ngắn hơn. Ví dụ `s1 = "abbaba"`, `s2 = "aba"`: vòng đầu tìm được `[0, 3]` = `abba`, nhưng cửa sổ tốt nhất là `[3, 5]` = `aba` — nó bắt đầu đúng tại `end = 3`. Quay về `start + 1` đảm bảo không bỏ sót.

In [9]:
def min_window_two_pointers(s1, s2):
    n, m = len(s1), len(s2)
    best_len, best_l = float('inf'), -1
    i = j = 0
    while i < n:
        if s1[i] == s2[j]:
            j += 1
            if j == m:                     # khớp xong tại end = i
                end = i
                j = m - 1
                while j >= 0:              # quét ngược
                    if s1[i] == s2[j]:
                        j -= 1
                    i -= 1
                i += 1                     # i = start
                if end - i + 1 < best_len:
                    best_len, best_l = end - i + 1, i
                j = 0                      # bắt đầu lại từ start + 1
        i += 1
    return "" if best_l == -1 else s1[best_l:best_l + best_len]

for s1, s2, exp in EXAMPLES:
    print(min_window_two_pointers(s1, s2), min_window_two_pointers(s1, s2) == exp)

bcde True
 True


**Code nộp LeetCode (Cách 3).**

```python
class Solution:
    def minWindow(self, s1: str, s2: str) -> str:
        n, m = len(s1), len(s2)
        best_len, best_l = float('inf'), -1
        i = j = 0
        while i < n:
            if s1[i] == s2[j]:
                j += 1
                if j == m:
                    end = i
                    j = m - 1
                    while j >= 0:
                        if s1[i] == s2[j]:
                            j -= 1
                        i -= 1
                    i += 1
                    if end - i + 1 < best_len:
                        best_len, best_l = end - i + 1, i
                    j = 0
            i += 1
        return '' if best_l == -1 else s1[best_l:best_l + best_len]
```

**Độ phức tạp.** Mỗi lần tìm được cửa sổ, ta lùi `i` về `start + 1`, nên một ký tự có thể bị quét nhiều lần. Trường hợp xấu `O(n·m)` (mỗi vòng xuôi–ngược dài cỡ `O(m)` nếu cửa sổ ngắn… và tối đa `O(n^2)` trong ca bệnh hiếm). Thực tế rất nhanh. Space `O(1)`.

### 7.3. Cách 4 — Quy hoạch động `dp[j]` = điểm bắt đầu muộn nhất

**Giải thích.** Duyệt `i` từ trái sang phải. Giữ mảng `dp` dài `m`:

- `dp[j]` = chỉ số bắt đầu **lớn nhất** `st` sao cho `s2[0..j]` là dãy con của `s1[st..i]`; `-1` nếu chưa có.

Khi xét `s1[i]`, duyệt `j` **từ cuối về đầu** (để không dùng `dp[j-1]` vừa cập nhật ở cùng bước `i` — giống knapsack 0/1):

- Nếu `s1[i] == s2[j]`: `dp[j] = i` nếu `j == 0`, ngược lại `dp[j] = dp[j-1]`.

Sau đó, nếu `dp[m-1] != -1`, cửa sổ `[dp[m-1], i]` là ứng viên.

Vì sao "muộn nhất" là đủ? Với cùng điểm kết thúc, start càng muộn cửa sổ càng ngắn; và giá trị `dp` chỉ tăng theo thời gian.

In [10]:
def min_window_dp(s1, s2):
    n, m = len(s1), len(s2)
    dp = [-1] * m
    best_len, best_l = float('inf'), -1
    for i, ch in enumerate(s1):
        for j in range(m - 1, -1, -1):
            if ch == s2[j]:
                dp[j] = i if j == 0 else dp[j - 1]
        if dp[m - 1] != -1 and i - dp[m - 1] + 1 < best_len:
            best_len, best_l = i - dp[m - 1] + 1, dp[m - 1]
    return "" if best_l == -1 else s1[best_l:best_l + best_len]

for s1, s2, exp in EXAMPLES:
    print(min_window_dp(s1, s2), min_window_dp(s1, s2) == exp)

bcde True
 True


> Vì `i` tăng dần và chỉ cập nhật khi **nhỏ hơn hẳn**, hai cửa sổ cùng độ dài thì cửa sổ kết thúc sớm hơn (cũng là bắt đầu sớm hơn) được giữ → đúng yêu cầu "bên trái nhất".

**Code nộp LeetCode (Cách 4).**

```python
class Solution:
    def minWindow(self, s1: str, s2: str) -> str:
        m = len(s2)
        dp = [-1] * m
        best_len, best_l = float('inf'), -1
        for i, ch in enumerate(s1):
            for j in range(m - 1, -1, -1):
                if ch == s2[j]:
                    dp[j] = i if j == 0 else dp[j - 1]
            if dp[-1] != -1 and i - dp[-1] + 1 < best_len:
                best_len, best_l = i - dp[-1] + 1, dp[-1]
        return '' if best_l == -1 else s1[best_l:best_l + best_len]
```

**Độ phức tạp.** Time `O(n·m)` = `2·10^4 × 100 = 2·10^6` — chắc chắn qua. Space `O(m)`.

**Tối ưu nhỏ.** Chỉ những `j` có `s2[j] == ch` mới cần xét: tiền xử lý `pos[ch]` = danh sách `j` (giảm dần). Khi đó mỗi `i` chỉ tốn `O(số lần ch xuất hiện trong s2)`.

In [11]:
def min_window_dp_fast(s1, s2):
    m = len(s2)
    pos = {}
    for j in range(m - 1, -1, -1):        # lưu j giảm dần
        pos.setdefault(s2[j], []).append(j)
    dp = [-1] * m
    best_len, best_l = float('inf'), -1
    for i, ch in enumerate(s1):
        for j in pos.get(ch, ()):
            dp[j] = i if j == 0 else dp[j - 1]
        if dp[-1] != -1 and i - dp[-1] + 1 < best_len:
            best_len, best_l = i - dp[-1] + 1, dp[-1]
    return "" if best_l == -1 else s1[best_l:best_l + best_len]

for s1, s2, exp in EXAMPLES:
    print(min_window_dp_fast(s1, s2) == exp)

True
True


### 7.4. Cách 5 — Bảng vị trí kế tiếp (next-occurrence table)

**Giải thích.** Tiền xử lý `nxt[i][c]` = chỉ số nhỏ nhất `k ≥ i` với `s1[k] == c`, hoặc `n` nếu không có. Xây từ phải sang trái: `nxt[i] = nxt[i+1]` rồi sửa `nxt[i][s1[i]] = i`.

Với mỗi `l` mà `s1[l] == s2[0]`: đặt `p = l`, rồi lần lượt `p = nxt[p+1][s2[j]]` cho `j = 1..m-1`. Nếu `p == n` thì từ `l` trở đi không còn cửa sổ (và các `l` lớn hơn cũng không) → dừng hẳn.

Mỗi điểm bắt đầu chỉ tốn `m` lần tra bảng.

In [12]:
def min_window_next_table(s1, s2):
    n, m = len(s1), len(s2)
    nxt = [None] * (n + 1)
    nxt[n] = [n] * 26
    for i in range(n - 1, -1, -1):
        row = nxt[i + 1][:]
        row[ord(s1[i]) - 97] = i
        nxt[i] = row
    best_len, best_l = float('inf'), -1
    first = ord(s2[0]) - 97
    l = nxt[0][first]
    while l < n:
        p = l
        for j in range(1, m):
            p = nxt[p + 1][ord(s2[j]) - 97]
            if p == n:
                break
        if p == n:
            break                       # từ l trở đi không còn cửa sổ
        if p - l + 1 < best_len:
            best_len, best_l = p - l + 1, l
        l = nxt[l + 1][first]
    return "" if best_l == -1 else s1[best_l:best_l + best_len]

for s1, s2, exp in EXAMPLES:
    print(min_window_next_table(s1, s2), min_window_next_table(s1, s2) == exp)

bcde True
 True


**Code nộp LeetCode (Cách 5).**

```python
class Solution:
    def minWindow(self, s1: str, s2: str) -> str:
        n, m = len(s1), len(s2)
        nxt = [None] * (n + 1)
        nxt[n] = [n] * 26
        for i in range(n - 1, -1, -1):
            row = nxt[i + 1][:]
            row[ord(s1[i]) - 97] = i
            nxt[i] = row
        t = [ord(c) - 97 for c in s2]
        best_len, best_l = float('inf'), -1
        l = nxt[0][t[0]]
        while l < n:
            p = l
            for j in range(1, m):
                p = nxt[p + 1][t[j]]
                if p == n:
                    break
            if p == n:
                break
            if p - l + 1 < best_len:
                best_len, best_l = p - l + 1, l
            l = nxt[l + 1][t[0]]
        return '' if best_l == -1 else s1[best_l:best_l + best_len]
```

**Độ phức tạp.** Tiền xử lý `O(26·n)` time và space; truy vấn `O(n·m)` trường hợp xấu, thường nhanh hơn nhiều. Space `O(26·n)` ≈ `5·10^5` số — chấp nhận được.

## 8. TRACE từng bước

### 8.1. Trace Cách 4 (DP) với `s1 = "abcdebdde"`, `s2 = "bde"`

`dp = [dp_b, dp_d, dp_e]`.

| i | s1[i] | dp sau bước | Ứng viên | Đáp án |
|---|---|---|---|---|
| 0 | a | [-1,-1,-1] | — | — |
| 1 | b | [1,-1,-1] | — | — |
| 2 | c | [1,-1,-1] | — | — |
| 3 | d | [1,1,-1] | — | — |
| 4 | e | [1,1,1] | [1,4] dài 4 | (4,1) |
| 5 | b | [5,1,1] | [1,5] dài 5 | (4,1) |
| 6 | d | [5,5,1] | [1,6] | (4,1) |
| 7 | d | [5,5,1] | [1,7] | (4,1) |
| 8 | e | [5,5,5] | [5,8] dài 4 — không nhỏ hơn hẳn | (4,1) |

In [13]:
def trace_dp(s1, s2):
    m = len(s2)
    dp = [-1] * m
    best = (float('inf'), -1)
    print(f"{'i':>2} {'ch':>2}  {'dp':<14} {'ứng viên':<12} best")
    for i, ch in enumerate(s1):
        for j in range(m - 1, -1, -1):
            if ch == s2[j]:
                dp[j] = i if j == 0 else dp[j - 1]
        cand = ""
        if dp[-1] != -1:
            cand = f"[{dp[-1]},{i}] len {i - dp[-1] + 1}"
            if i - dp[-1] + 1 < best[0]:
                best = (i - dp[-1] + 1, dp[-1])
        print(f"{i:>2} {ch:>2}  {str(dp):<14} {cand:<12} {best}")
    return "" if best[1] == -1 else s1[best[1]:best[1] + best[0]]

print("Kết quả:", trace_dp("abcdebdde", "bde"))

 i ch  dp             ứng viên     best
 0  a  [-1, -1, -1]                (inf, -1)
 1  b  [1, -1, -1]                 (inf, -1)
 2  c  [1, -1, -1]                 (inf, -1)
 3  d  [1, 1, -1]                  (inf, -1)
 4  e  [1, 1, 1]      [1,4] len 4  (4, 1)
 5  b  [5, 1, 1]      [1,5] len 5  (4, 1)
 6  d  [5, 5, 1]      [1,6] len 6  (4, 1)
 7  d  [5, 5, 1]      [1,7] len 7  (4, 1)
 8  e  [5, 5, 5]      [5,8] len 4  (4, 1)
Kết quả: bcde


### 8.2. Trace Cách 3 (xuôi – ngược)

Vòng 1: xuôi từ 0 khớp b@1, d@3, e@4 → `end = 4`. Ngược từ 4: e@4, d@3, b@1 → `start = 1`. Cửa sổ `[1,4]` = `bcde`. Quay về `i = 2`.

Vòng 2: xuôi từ 2: b@5, d@6, e@8 → `end = 8`. Ngược: e@8, d@7 (gặp d muộn nhất!), b@5 → `start = 5`. Cửa sổ `[5,8]` = `bdde`, dài 4, không nhỏ hơn hẳn. Quay về `i = 6`: không còn `b` → kết thúc.

In [14]:
def trace_two_pointers(s1, s2):
    n, m = len(s1), len(s2)
    best = (float('inf'), -1)
    i = j = 0
    while i < n:
        if s1[i] == s2[j]:
            j += 1
            if j == m:
                end = i
                j = m - 1
                path = []
                while j >= 0:
                    if s1[i] == s2[j]:
                        path.append(f"{s1[i]}@{i}")
                        j -= 1
                    i -= 1
                i += 1
                print(f"xuôi tới end={end}; ngược khớp {path} -> start={i}, cửa sổ {s1[i:end+1]!r}")
                if end - i + 1 < best[0]:
                    best = (end - i + 1, i)
                j = 0
        i += 1
    return "" if best[1] == -1 else s1[best[1]:best[1] + best[0]]

print("Kết quả:", trace_two_pointers("abcdebdde", "bde"))

xuôi tới end=4; ngược khớp ['e@4', 'd@3', 'b@1'] -> start=1, cửa sổ 'bcde'
xuôi tới end=8; ngược khớp ['e@8', 'd@7', 'b@5'] -> start=5, cửa sổ 'bdde'
Kết quả: bcde


In [15]:
# Trace thêm: một ví dụ cho thấy quét ngược giúp thu hẹp cửa sổ
print("Kết quả:", trace_two_pointers("abbbcbdbdee", "bde"))

xuôi tới end=9; ngược khớp ['e@9', 'd@8', 'b@7'] -> start=7, cửa sổ 'bde'
Kết quả: bde


## 9. Code nộp LeetCode

Chọn **Cách 4 (DP)** làm bản chính: ngắn, `O(n·m)` chắc chắn, dễ chứng minh.

In [16]:
class Solution:
    def minWindow(self, s1: str, s2: str) -> str:
        m = len(s2)
        dp = [-1] * m
        best_len, best_l = float('inf'), -1
        for i, ch in enumerate(s1):
            for j in range(m - 1, -1, -1):
                if ch == s2[j]:
                    dp[j] = i if j == 0 else dp[j - 1]
            if dp[-1] != -1 and i - dp[-1] + 1 < best_len:
                best_len, best_l = i - dp[-1] + 1, dp[-1]
        return '' if best_l == -1 else s1[best_l:best_l + best_len]

for s1, s2, exp in EXAMPLES:
    assert Solution().minWindow(s1, s2) == exp
print("Solution OK trên ví dụ đề")

Solution OK trên ví dụ đề


## 10. Vì sao thuật toán đúng

**Bất biến DP.** Sau khi xử lý `s1[0..i]`, với mọi `j`, `dp[j]` là chỉ số `st` **lớn nhất** sao cho `s2[0..j]` là dãy con của `s1[st..i]` (hoặc `-1` nếu không tồn tại).

*Chứng minh quy nạp theo `i`.* Xét cách khớp `s2[0..j]` trong `s1[st..i]` với `st` lớn nhất.

- Nếu cách khớp không dùng `s1[i]` thì nó nằm trong `s1[st..i-1]` → `st ≤ dp_cũ[j]`; và ngược lại `dp_cũ[j]` vẫn hợp lệ cho `i`. Vậy `st = dp_cũ[j]`.
- Nếu dùng `s1[i]` cho `s2[j]` (cần `s1[i] == s2[j]`): phần `s2[0..j-1]` khớp trong `s1[st..i-1]` → `st` lớn nhất chính là `dp_cũ[j-1]` (với `j = 0`, `st = i`).

Giá trị mới = max của hai trường hợp. Vì `dp_cũ[j-1] ≥ dp_cũ[j]` (khớp ít ký tự hơn thì bắt đầu muộn hơn được) nên khi `s1[i] == s2[j]`, max chính là `dp_cũ[j-1]` — đúng công thức. Duyệt `j` giảm dần đảm bảo ta dùng `dp_cũ[j-1]` (chưa bị ghi đè ở bước `i`).

**Đáp án.** Mọi cửa sổ tối ưu `[l, r]` có `l ≤ dp[m-1]` tại thời điểm `r`; cửa sổ `[dp[m-1], r]` ngắn nhất trong các cửa sổ kết thúc tại `r`. Lấy min theo `r` tăng dần với so sánh nghiêm ngặt → được cửa sổ ngắn nhất, và trong các cửa sổ cùng độ dài, cái có `r` (do đó `l`) nhỏ nhất.

**Cách 3** đúng vì: quét ngược tham lam từ `end` cho start muộn nhất (đối xứng với mục 3.2); và mọi cửa sổ tối ưu bắt đầu ở `l' > start` sẽ được tìm thấy khi ta quay lại quét từ `start + 1`.

## 11. Phân tích Time/Space

`n = |s1| ≤ 2·10^4`, `m = |s2| ≤ 100`.

| Cách | Time | Space |
|---|---|---|
| 1. Mọi cửa sổ | `O(n^3)` | `O(n)` (lát cắt) |
| 2. Tham lam từ mỗi start | `O(n^2)` | `O(1)` |
| 3. Xuôi–ngược | `O(n·m)` điển hình, xấu `O(n^2)` | `O(1)` |
| 4. DP | `O(n·m)` | `O(m)` |
| 5. Bảng kế tiếp | `O(26n + n·m)` | `O(26n)` |

Thử nghiệm dưới đây dùng ca xấu cho Cách 2: `s1` toàn `a`, `s2 = "a"*50 + "b"`.

In [17]:
import time, random
s1 = "a" * 3000 + "b"
s2 = "a" * 50 + "b"
for fn in [min_window_greedy_start, min_window_two_pointers, min_window_dp, min_window_dp_fast, min_window_next_table]:
    t = time.perf_counter(); r = fn(s1, s2); t = time.perf_counter() - t
    print(f"{fn.__name__:<26} {t*1000:8.2f} ms  len={len(r)}")

min_window_greedy_start      230.19 ms  len=51
min_window_two_pointers        0.13 ms  len=51
min_window_dp                  4.88 ms  len=51
min_window_dp_fast             3.67 ms  len=51
min_window_next_table          7.99 ms  len=51


## 12. Bộ test

In [18]:
ALL = [min_window_brute, min_window_greedy_start, min_window_two_pointers,
       min_window_dp, min_window_dp_fast, min_window_next_table, Solution().minWindow]

EDGE = [
    ("a", "a", "a"),
    ("a", "b", ""),
    ("ab", "ba", ""),            # sai thứ tự
    ("abc", "abc", "abc"),
    ("abc", "abcd", ""),         # s2 dài hơn s1
    ("aaaa", "aa", "aa"),        # leftmost
    ("cnhczmccqouqadqtmjjzl", "mm", "mccqouqadqtm"),
    ("fgrqsqsnodwmxzkzxwqegkndaa", "kzed", "kzxwqegknd"),
    ("abcdebdde", "bde", "bcde"),
]
for fn in ALL:
    for s1, s2, exp in EXAMPLES + EDGE:
        assert fn(s1, s2) == exp, (fn.__name__, s1, s2, fn(s1, s2), exp)
print("Ví dụ đề + edge case OK cho", len(ALL), "cách")

Ví dụ đề + edge case OK cho 7 cách


In [19]:
rng = random.Random(727)
for _ in range(3000):
    n = rng.randint(1, 14)
    alpha = "abc"[:rng.randint(1, 3)]
    s1 = "".join(rng.choice(alpha) for _ in range(n))
    s2 = "".join(rng.choice(alpha) for _ in range(rng.randint(1, 4)))
    ref = min_window_brute(s1, s2)
    for fn in ALL[1:]:
        assert fn(s1, s2) == ref, (fn.__name__, s1, s2, fn(s1, s2), ref)
print("3000 test ngẫu nhiên: mọi cách khớp brute force")

3000 test ngẫu nhiên: mọi cách khớp brute force


In [20]:
# Test lớn: các cách nhanh đối chiếu với nhau
for _ in range(5):
    s1 = "".join(rng.choice("ab") for _ in range(20000))
    s2 = "".join(rng.choice("ab") for _ in range(100))
    res = {fn.__name__: fn(s1, s2) for fn in [min_window_two_pointers, min_window_dp_fast, min_window_next_table]}
    assert len(set(res.values())) == 1
print("Test lớn n=20000, m=100 khớp nhau")

Test lớn n=20000, m=100 khớp nhau


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Dùng sliding window đếm ký tự như bài 76

**Code sai:** chỉ kiểm tra cửa sổ có **đủ số lượng** ký tự của `s2`.

**Vì sao sai:** bỏ qua thứ tự. Với `s1 = "deb"`, `s2 = "bde"`: cửa sổ `"deb"` đủ ký tự nhưng không chứa `bde` theo thứ tự.

**Code đúng:** kiểm tra dãy con (DP hoặc xuôi–ngược).

In [21]:
from collections import Counter
def window_by_count_wrong(s1, s2):
    need = Counter(s2)
    best = ""
    for l in range(len(s1)):
        for r in range(l, len(s1)):
            if not need - Counter(s1[l:r + 1]):
                if not best or r - l + 1 < len(best):
                    best = s1[l:r + 1]
                break
    return best

print("sai:", repr(window_by_count_wrong("deb", "bde")), " đúng:", repr(Solution().minWindow("deb", "bde")))

sai: 'deb'  đúng: ''


### Lỗi 2 — Duyệt `j` tăng dần trong DP

**Code sai:** `for j in range(m)` → khi `s2 = "aa"`, một ký tự `a` của `s1` được dùng cho **cả** `s2[0]` lẫn `s2[1]`.

**Code đúng:** duyệt `j` **giảm dần** (giống knapsack 0/1).

In [22]:
def dp_wrong(s1, s2):
    m = len(s2); dp = [-1] * m; best = (float('inf'), -1)
    for i, ch in enumerate(s1):
        for j in range(m):                       # SAI: tăng dần
            if ch == s2[j]:
                dp[j] = i if j == 0 else dp[j - 1]
        if dp[-1] != -1 and i - dp[-1] + 1 < best[0]:
            best = (i - dp[-1] + 1, dp[-1])
    return "" if best[1] == -1 else s1[best[1]:best[1] + best[0]]

print("sai:", repr(dp_wrong("ab", "aa")), " đúng:", repr(Solution().minWindow("ab", "aa")))

sai: 'a'  đúng: ''


### Lỗi 3 — Cập nhật khi `<=` làm mất "bên trái nhất"

**Code sai:** `if length <= best_len` → cửa sổ cùng độ dài nhưng xuất hiện sau ghi đè.

**Code đúng:** chỉ cập nhật khi `<` (với thứ tự duyệt tăng dần).

In [23]:
def dp_le_wrong(s1, s2):
    m = len(s2); dp = [-1] * m; best = (float('inf'), -1)
    for i, ch in enumerate(s1):
        for j in range(m - 1, -1, -1):
            if ch == s2[j]:
                dp[j] = i if j == 0 else dp[j - 1]
        if dp[-1] != -1 and i - dp[-1] + 1 <= best[0]:   # SAI: <=
            best = (i - dp[-1] + 1, dp[-1])
    return "" if best[1] == -1 else s1[best[1]:best[1] + best[0]]

print("sai:", dp_le_wrong("abcdebdde", "bde"), " đúng:", Solution().minWindow("abcdebdde", "bde"))

sai: bdde  đúng: bcde


### Lỗi 4 — Quét xuôi–ngược nhưng tiếp tục từ `end + 1`

**Code sai:** sau khi tìm cửa sổ `[start, end]`, tiếp tục từ `end + 1`.

**Vì sao sai:** có thể bỏ sót cửa sổ bắt đầu trong `(start, end]`. Ví dụ `s1 = "abbaba"`, `s2 = "aba"`: cửa sổ đầu là `[0,3]`; nếu nhảy tới `end + 1 = 4` thì ký tự `a` tại vị trí 3 không bao giờ được dùng làm `s2[0]` nữa, và ta bỏ lỡ `[3,5] = "aba"`. Cell dưới kiểm tra ca này và tìm thêm phản ví dụ ngẫu nhiên.

**Code đúng:** tiếp tục từ `start + 1`.

In [24]:
def two_pointers_wrong(s1, s2):
    n, m = len(s1), len(s2)
    best = (float('inf'), -1)
    i = j = 0
    while i < n:
        if s1[i] == s2[j]:
            j += 1
            if j == m:
                end = i; j = m - 1
                while j >= 0:
                    if s1[i] == s2[j]: j -= 1
                    i -= 1
                start = i + 1
                if end - start + 1 < best[0]:
                    best = (end - start + 1, start)
                i, j = end, 0                       # SAI: tiếp tục sau end
        i += 1
    return "" if best[1] == -1 else s1[best[1]:best[1] + best[0]]

print("abbaba/aba -> sai:", two_pointers_wrong("abbaba", "aba"), " đúng:", Solution().minWindow("abbaba", "aba"))
rng2 = random.Random(1)
for _ in range(10000):
    s1 = "".join(rng2.choice("ab") for _ in range(rng2.randint(1, 8)))
    s2 = "".join(rng2.choice("ab") for _ in range(rng2.randint(1, 3)))
    if two_pointers_wrong(s1, s2) != min_window_brute(s1, s2):
        print("Phản ví dụ:", s1, s2, "sai:", two_pointers_wrong(s1, s2), "đúng:", min_window_brute(s1, s2))
        break

abbaba/aba -> sai: abba  đúng: aba
Phản ví dụ: babbaa aa sai: abba đúng: aa


## 14. Mẹo & nhận dạng pattern

- **"Cửa sổ ngắn nhất chứa … theo thứ tự"** → không phải sliding window bộ đếm; nghĩ tới **DP điểm bắt đầu muộn nhất** hoặc **xuôi–ngược**.
- **Dãy con + tham lam**: luôn dùng ký tự khớp **sớm nhất** (xuôi) hoặc **muộn nhất** (ngược).
- **DP 1 chiều cập nhật tại chỗ** khi mỗi phần tử chỉ được dùng một lần → duyệt chỉ số **giảm dần**.
- **Nhiều truy vấn dãy con trên cùng một chuỗi lớn** (ví dụ *Is Subsequence (392)* follow-up, *Number of Matching Subsequences (792)*) → **bảng vị trí kế tiếp** hoặc `bisect` trên danh sách vị trí.
- Ưu tiên "bên trái nhất": duyệt tăng dần + cập nhật bằng `<`.

## 15. Biến thể

1. **Minimum Window Substring (76)**: chứa đủ ký tự (multiset) — dùng sliding window + bộ đếm.
2. **Đếm số cửa sổ** (không chỉ cửa sổ ngắn nhất) chứa `s2` làm dãy con: với mỗi `i`, số start hợp lệ là `dp[m-1] + 1`.
3. **Trả về độ dài** thay vì chuỗi; hoặc trả về **tất cả** cửa sổ tối thiểu.
4. **Nhiều `s2`** với cùng `s1`: dùng bảng vị trí kế tiếp xây một lần.
5. **Is Subsequence (392)**: phiên bản kiểm tra đơn giản nhất.

In [25]:
# Biến thể 2: đếm số đoạn con của s1 chứa s2 làm dãy con
def count_windows(s1, s2):
    m = len(s2); dp = [-1] * m; total = 0
    for i, ch in enumerate(s1):
        for j in range(m - 1, -1, -1):
            if ch == s2[j]:
                dp[j] = i if j == 0 else dp[j - 1]
        total += dp[-1] + 1          # mọi start trong [0, dp[-1]] đều hợp lệ
    return total

def count_windows_brute(s1, s2):
    n = len(s1)
    return sum(is_subsequence(s2, s1[l:r + 1]) for l in range(n) for r in range(l, n))

for s1, s2 in [("abcdebdde", "bde"), ("aaaa", "aa"), ("abc", "d")]:
    print(s1, s2, count_windows(s1, s2), count_windows_brute(s1, s2))

abcdebdde bde 14 14
aaaa aa 6 6
abc d 0 0


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi cửa sổ | Thử `(l, r)`, kiểm tra dãy con | `O(n^3)` | `O(n)` | Trọng tài |
| 2. Tham lam mỗi start | Khớp xuôi từ mỗi `l` | `O(n^2)` | `O(1)` | Bước trung gian để hiểu |
| 3. Xuôi–ngược | Khớp xuôi tìm end, ngược tìm start | `O(n·m)` điển hình | `O(1)` | Phỏng vấn, bộ nhớ ít |
| 4. DP start muộn nhất | `dp[j] = dp[j-1]` khi khớp | `O(n·m)` | `O(m)` | Bản nộp chính, dễ chứng minh |
| 5. Bảng kế tiếp | Nhảy `m` bước mỗi start | `O(26n + n·m)` | `O(26n)` | Nhiều truy vấn trên cùng `s1` |

## 17. Bài tập

**Bài 1 (đoán output).** `minWindow("abcbcbca", "bca")` = ?

**Bài 2 (tính tay).** Lập bảng `dp` (như 8.1) cho `s1 = "aabba"`, `s2 = "ab"`. Đáp án là gì?

**Bài 3 (điền TODO).** Hoàn thiện `backward_start(s1, s2, end)` trả về start muộn nhất sao cho `s2` là dãy con của `s1[start..end]` (biết `s1[end] == s2[-1]` và tồn tại).

**Bài 4 (tự viết lại).** Viết lại Cách 3 (xuôi–ngược) trong hàm `my_min_window(s1, s2)`.

**Bài 5 (mini challenge).** Viết `min_window_len_all(s1, s2)` trả về **danh sách mọi** cửa sổ `(start, end)` có độ dài nhỏ nhất (không chỉ cửa sổ trái nhất).

In [26]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def backward_start(s1, s2, end):
    # TODO: j = len(s2) - 1; i = end; lùi i, giảm j khi khớp; trả i khi j < 0
    pass

# Bài 4 — TODO
def my_min_window(s1, s2):
    # TODO: vòng while i < n; khi j == m thì quét ngược, cập nhật, j = 0
    pass

# Bài 5 — TODO
def min_window_len_all(s1, s2):
    # TODO: chạy DP, lưu mọi (dp[-1], i) rồi lọc độ dài nhỏ nhất (loại trùng)
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: liệt kê các cửa sổ bắt đầu bằng `b`.
- Mức 2: start 1: b c (b) c b c a → khớp b@1, c@2, a@7 → dài 7; start 3: b@3 c@4 a@7 → 5; start 5: b@5 c@6 a@7 → 3.
- Mức 3: `"bca"`.

**Bài 2.**
- Mức 1: `dp = [dp_a, dp_b]`.
- Mức 2: i=0: [0,-1]; i=1: [1,-1]; i=2 (b): [1,1] → cửa sổ [1,2].
- Mức 3: i=3 (b): [1,1] → [1,3] dài hơn; đáp án `"ab"`.

**Bài 3.**
- Mức 1: so khớp từ cuối `s2`.
- Mức 2: mỗi khi khớp thì `j -= 1`; luôn `i -= 1`.
- Mức 3: khi `j < 0` thì vị trí vừa khớp là `i + 1`.

**Bài 4.**
- Mức 1: `i, j = 0, 0`.
- Mức 2: khi `j == m`: lưu `end`, quét ngược tìm `start`.
- Mức 3: đặt `i = start` (vì cuối vòng có `i += 1`), `j = 0`.

**Bài 5.**
- Mức 1: tại mỗi `i`, cửa sổ tốt nhất kết thúc ở `i` là `[dp[-1], i]`.
- Mức 2: cửa sổ tối thiểu toàn cục phải là cửa sổ tốt nhất tại `end` của nó.
- Mức 3: thu thập, tìm độ dài nhỏ nhất, lọc, bỏ trùng bằng `set`.

## 19. Đáp án tham khảo

In [27]:
# Bài 1, 2
assert Solution().minWindow("abcbcbca", "bca") == "bca"
assert Solution().minWindow("aabba", "ab") == "ab"

# Bài 3
def backward_start_ans(s1, s2, end):
    j, i = len(s2) - 1, end
    while j >= 0:
        if s1[i] == s2[j]:
            j -= 1
        i -= 1
    return i + 1

assert backward_start_ans("abcdebdde", "bde", 8) == 5
assert backward_start_ans("abcdebdde", "bde", 4) == 1
print("Bài 1–3 OK")

Bài 1–3 OK


In [28]:
# Bài 4
def my_min_window_ans(s1, s2):
    n, m = len(s1), len(s2)
    best = (float('inf'), -1)
    i = j = 0
    while i < n:
        if s1[i] == s2[j]:
            j += 1
            if j == m:
                end = i
                start = backward_start_ans(s1, s2, end)
                if end - start + 1 < best[0]:
                    best = (end - start + 1, start)
                i, j = start, 0
        i += 1
    return "" if best[1] == -1 else s1[best[1]:best[1] + best[0]]

# Bài 5
def min_window_len_all_ans(s1, s2):
    m = len(s2); dp = [-1] * m; cands = set()
    for i, ch in enumerate(s1):
        for j in range(m - 1, -1, -1):
            if ch == s2[j]:
                dp[j] = i if j == 0 else dp[j - 1]
        if dp[-1] != -1:
            cands.add((dp[-1], i))
    if not cands:
        return []
    L = min(e - s for s, e in cands)
    return sorted((s, e) for s, e in cands if e - s == L)

def all_brute(s1, s2):
    n = len(s1); res = []
    for l in range(n):
        for r in range(l, n):
            if is_subsequence(s2, s1[l:r + 1]):
                res.append((l, r))
    if not res:
        return []
    L = min(r - l for l, r in res)
    return [(l, r) for l, r in res if r - l == L]

rng = random.Random(9)
for _ in range(1000):
    s1 = "".join(rng.choice("abc") for _ in range(rng.randint(1, 12)))
    s2 = "".join(rng.choice("abc") for _ in range(rng.randint(1, 3)))
    assert my_min_window_ans(s1, s2) == min_window_brute(s1, s2)
    assert min_window_len_all_ans(s1, s2) == all_brute(s1, s2)
print(min_window_len_all_ans("abcdebdde", "bde"))
print("Bài 4–5 OK")

[(1, 4), (5, 8)]
Bài 4–5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template DP "start muộn nhất":**

```python
dp = [-1] * m
for i, ch in enumerate(s1):
    for j in range(m - 1, -1, -1):          # GIẢM DẦN
        if ch == s2[j]:
            dp[j] = i if j == 0 else dp[j - 1]
    if dp[-1] != -1 and i - dp[-1] + 1 < best_len:   # NGHIÊM NGẶT
        best_len, best_l = i - dp[-1] + 1, dp[-1]
```

**Template xuôi–ngược:** xuôi đến khi `j == m` → `end`; ngược từ `end` đến khi `j < 0` → `start`; cập nhật; tiếp tục từ `start + 1`.

**Checklist trước khi submit:**

- [ ] Dãy con (giữ thứ tự), **không** phải đếm ký tự.
- [ ] DP duyệt `j` giảm dần.
- [ ] Cập nhật đáp án bằng `<` để giữ cửa sổ trái nhất.
- [ ] Không có cửa sổ → trả `""`.
- [ ] `s2` dài hơn `s1` → `""` (code tự xử lý).
- [ ] Xuôi–ngược: tiếp tục từ `start + 1`, không phải `end + 1`.
- [ ] Độ phức tạp `O(n·m) = 2·10^6` — an toàn.